# Chapter 19: state space and Kalman filtering

Check scalar and bivariate conditional-Gaussian updates, then use fixed-seed latent alpha/beta paths to inspect process-noise misspecification. Q, R and initial states are given, with no parameter estimation or smoother. Save constructed_latent.csv and constructed_filter.csv under RUN_DIR. Separately display two frozen constructed reference figures without regenerating them. The French branch is disabled.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch19_state_space_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys

sys.path.insert(0, str(ROOT / 'code'))
import ch19_state_space as workflow
from quant_course.state_space import kalman_filter
import numpy as np
from IPython.display import Image, display
print('Project root:', ROOT)

## Prior, innovation, gain and posterior

Prior beta is 1 with variance 0.05. Monthly market excess return is 0.02, asset excess return 0.03 and observation variance 0.0001. The predicted observation is 0.02, innovation 0.01 and innovation variance 0.00012. Gain 8.3333 has beta/return units, not probability units.

In [ ]:
scalar = kalman_filter(
    [.03], [[.02]], initial_state=[1.0], initial_covariance=[[.04]],
    transition=[[1.0]], process_covariance=[[.01]], observation_variance=.0001,
)
expected = [1.0, .05, .01, .00012, 8.333333333333333, 1.0833333333333333, .04166666666666667]
actual = [scalar.predicted_states[0, 0], scalar.predicted_covariances[0, 0, 0],
          scalar.innovations[0], scalar.innovation_variances[0], scalar.gains[0, 0],
          scalar.filtered_states[0, 0], scalar.filtered_covariances[0, 0, 0]]
np.testing.assert_allclose(actual, expected)
print('prior beta, prior variance, innovation, S, K, posterior beta, posterior variance')
print(actual)

## Two-state conditional Gaussian check

Prior alpha=0 and beta=1, covariance diag(0.0004,0.04), design [1,0.02], observation 0.03 and noise variance 0.0001 give innovation variance 0.000516 and posterior mean approximately (0.007752,1.015504). Compare the formula with the core filter.

In [ ]:
P = np.diag([.0004, .04])
h = np.array([1.0, .02])
prior_mean = np.array([0.0, 1.0])
y, R = .03, .0001
S = float(h @ P @ h + R)
cross = P @ h
innovation = y - h @ prior_mean
gain = cross / S
conditional_mean = prior_mean + gain * innovation
conditional_covariance = P - np.outer(cross, cross) / S
two_state = kalman_filter([y], [h], initial_state=prior_mean,
                          initial_covariance=P, transition=np.eye(2),
                          process_covariance=np.zeros((2, 2)), observation_variance=R)
np.testing.assert_allclose(S, .000516)
np.testing.assert_allclose(innovation, .01)
np.testing.assert_allclose(two_state.innovation_variances[0], S)
np.testing.assert_allclose(two_state.gains[0], gain)
np.testing.assert_allclose(two_state.filtered_states[0], conditional_mean)
np.testing.assert_allclose(two_state.filtered_covariances[0], conditional_covariance)
np.testing.assert_allclose(conditional_mean, [.0077519379845, 1.015503875969])
assert np.linalg.eigvalsh(conditional_covariance).min() >= 0
print('innovation =', innovation, '\nS =', S, '\nK =', gain,
      '\nposterior mean =', conditional_mean,
      '\nposterior covariance =\n', conditional_covariance)

## Fixed-seed mechanism and disabled French example

The simulation contains known latent alpha/beta and compares correct Q against zero beta-process variance. One path illustrates misspecification, not predictive superiority. The historical French example specifies its initial values and Q/R; its revised monthly history is not PIT and is not run here.

In [ ]:
simulated, simulation_filter, simulation_metrics = workflow.simulate_known_latent()
print("CONSTRUCTED", simulation_metrics)
simulated.to_csv(RUN_DIR / "constructed_latent.csv")
simulation_filter.to_csv(RUN_DIR / "constructed_filter.csv")
assert len(simulated) == workflow.SIMULATION_N
print("French filter branch SKIPPED: data rights unknown")


## Filtering and smoothing information

predicted_state[t] uses observations through t−1; filtered_state[t] also uses observation t. A smoothed state would condition on later observations, but the core has no smoother. A displayed image is a frozen CLI asset rather than a newly saved filter result.

In [ ]:
figure_names = [
    'ch19_simulated_latent_beta.png',
    'ch19_simulated_innovation_update.png',
]
for name in figure_names:
    path = ROOT / 'fixtures/figures/a4' / name
    assert path.is_file(), f'Missing pre-generated figure: {path}'
    print(name)
    display(Image(filename=str(path)))